# ODI to Databricks Migration: TRG_LOC Direct Insert

**Source File:** TARGET ODI SQL file
**Conversion Timestamp:** 2024-07-30 12:00:00
**Description:** Direct insertion of data from the `LOCATIONS` table into `TRG_LOC`.

In [ ]:
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_odi_sess_no"))

# Direct Insert to Target Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_loc
(
  LOCATION_ID ,
  STREET_ADDRESS ,
  POSTAL_CODE ,
  CITY ,
  STATE_PROVINCE ,
  COUNTRY_ID
)
SELECT
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID
FROM
  workspace.hr.locations AS LOCATIONS;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS target_record_count FROM workspace.hr.trg_loc;

# Conversion Notes

1.  **Schema and Table Naming:** Original Oracle schema `HR` and table names `TRG_LOC`, `LOCATIONS` have been converted to lowercase and prefixed with `workspace.` (e.g., `workspace.hr.trg_loc`, `workspace.hr.locations`) as per migration guidelines.
2.  **Oracle Hints Removal:** The Oracle `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Databricks Delta Lake.
3.  **Data Types:** Column types are assumed to be compatible with Spark SQL's default inference or common mappings (e.g., Oracle `NUMBER` to `BIGINT`/`DECIMAL`, `VARCHAR2` to `STRING`). Please verify the actual DDL for `workspace.hr.trg_loc` and `workspace.hr.locations` to ensure correct type mapping if tables are created via DDL.
4.  **Missing DDL:** This notebook assumes the target table `workspace.hr.trg_loc` already exists. If not, a `CREATE TABLE` statement with appropriate schema and column definitions should precede the `INSERT` statement.